# Clone DiffDock

In [1]:
!git clone https://github.com/gcorso/DiffDock.git
%cd DiffDock

Cloning into 'DiffDock'...
remote: Enumerating objects: 520, done.
remote: Counting objects: 100% (293/293), done.
remote: Compressing objects: 100% (137/137), done.
remote: Total 520 (delta 209), reused 156 (delta 156), pack-reused 227 (from 3)
Receiving objects: 100% (520/520), 233.08 MiB | 17.58 MiB/s, done.
Resolving deltas: 100% (244/244), done.
/content/DiffDock


# Install Dependencies


In [2]:
!pip install -q torch torchvision torchaudio
!pip install -q torch-geometric
!pip install -q e3nn biopython spyrmsd
!pip install -q pandas scipy scikit-learn pyyaml tqdm networkx
!pip install -q rdkit
!pip install -q fair-esm
!pip install torch_cluster -f https://data.pyg.org/whl/torch-2.10.0+cu128.html
!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.10.0+cu128.html
!pip install -q prody
!pip install -q py3Dmol
!pip install biopandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 84.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 450.7/450.7 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 130.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 107.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 9.9 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.10.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 95.3 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.10.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 55.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 82.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 23.1 MB/s eta 0:0

# Download Processed Datasets
PDBBind processed data has copyright infringement, currently unavailable.
Here, we use PoseBusters instead.

In [3]:
# Download
!wget -O BindingMOAD_2020_processed.tar \
  "https://zenodo.org/records/10656052/files/BindingMOAD_2020_processed.tar?download=1"

--2026-04-12 14:39:13--  https://zenodo.org/records/10656052/files/BindingMOAD_2020_processed.tar?download=1
Resolving zenodo.org (zenodo.org)... 188.185.48.75, 188.184.98.114, 188.184.103.118, ...
Connecting to zenodo.org (zenodo.org)|188.185.48.75|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 28543702016 (27G) [application/octet-stream]
Saving to: ‘BindingMOAD_2020_processed.tar’

BindingMOAD_2020_pr 100%[===================>]  26.58G  15.7MB/s    in 28m 35s 

2026-04-12 15:07:49 (15.9 MB/s) - ‘BindingMOAD_2020_processed.tar’ saved [28543702016/28543702016]



In [4]:
# Extract
!mkdir -p data/BindingMOAD_2020_processed
!tar -xf BindingMOAD_2020_processed.tar -C data/BindingMOAD_2020_processed

# Inspect the folder structure
!find data/BindingMOAD_2020_processed -maxdepth 2 -type d | sort | head -50

Streaming output truncated to the last 5000 lines.
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.quarantine'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.quarantine'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.quarantine'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.quarantine'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.quarantine'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.quarantine'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.quarantine'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.quarantine'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.quarantine'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.quarantine'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.quarantine'
tar

In [5]:
# Generate the ESM2 embeddings
!cd /content/DiffDock \
&& PYTHONPATH=/content/DiffDock python datasets/esm_embedding_preparation.py \
  --data_dir /content/DiffDock/data/BindingMOAD_2020_processed/BindingMOAD_2020_processed/pdb_protein \
  --dataset moad


Streaming output truncated to the last 5000 lines.
Some atoms or residues may be missing in the data structure.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/Bio/PDB/PDBParser.py:403: PDBConstructionWarning: PDBConstructionException: Atom CE1 defined twice in residue <Residue HIS het=  resseq=297 icode= > at line 9569.
Exception ignored.
Some atoms or residues may be missing in the data structure.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/Bio/PDB/PDBParser.py:403: PDBConstructionWarning: PDBConstructionException: Atom NE2 defined twice in residue <Residue HIS het=  resseq=297 icode= > at line 9571.
Exception ignored.
Some atoms or residues may be missing in the data structure.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/Bio/PDB/StructureBuilder.py:137: PDBConstructionWarning: WARNING: Residue (' ', 298, ' ') redefined at line 9573.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/Bio/PDB/StructureBuilder.py:159: PDBConstructionWarning: WAR

In [ ]:
# Create a csv with path to all proteins and ligands
import os
import pandas as pd

root = "data/BindingMOAD_2020_processed/BindingMOAD_2020_processed"
protein_dir = os.path.join(root, "pdb_protein")
ligand_dir = os.path.join(root, "pdb_superligand")

rows = []

for protein_file in os.listdir(protein_dir):
    if protein_file.endswith(".pdb"):
        base = protein_file.replace(".pdb", "")

        protein_path = os.path.join(protein_dir, protein_file)

        # try common ligand extensions
        ligand_sdf = os.path.join(ligand_dir, f"{base}.sdf")
        ligand_mol2 = os.path.join(ligand_dir, f"{base}.mol2")
        ligand_pdb = os.path.join(ligand_dir, f"{base}.pdb")

        ligand_path = None
        if os.path.exists(ligand_sdf):
            ligand_path = ligand_sdf
        elif os.path.exists(ligand_mol2):
            ligand_path = ligand_mol2
        elif os.path.exists(ligand_pdb):
            ligand_path = ligand_pdb

        if ligand_path is not None:
            rows.append({
                "protein_path": protein_path,
                "ligand": ligand_path
            })

df = pd.DataFrame(rows)
df.to_csv("data/moad_for_esm.csv", index=False)

print(df.head())
print("Number of rows:", len(df))

In [ ]:
!head data/moad_for_esm.csv

protein_path,ligand
data/posebusters_benchmark_set/posebusters_benchmark_set/7RPZ_6IC/7RPZ_6IC_protein.pdb,data/posebusters_benchmark_set/posebusters_benchmark_set/7RPZ_6IC/7RPZ_6IC_ligands.sdf
data/posebusters_benchmark_set/posebusters_benchmark_set/7UP3_NZ0/7UP3_NZ0_protein.pdb,data/posebusters_benchmark_set/posebusters_benchmark_set/7UP3_NZ0/7UP3_NZ0_ligands.sdf
data/posebusters_benchmark_set/posebusters_benchmark_set/7NF0_BYN/7NF0_BYN_protein.pdb,data/posebusters_benchmark_set/posebusters_benchmark_set/7NF0_BYN/7NF0_BYN_ligands.sdf
data/posebusters_benchmark_set/posebusters_benchmark_set/7ZZB_KGX/7ZZB_KGX_protein.pdb,data/posebusters_benchmark_set/posebusters_benchmark_set/7ZZB_KGX/7ZZB_KGX_ligands.sdf
data/posebusters_benchmark_set/posebusters_benchmark_set/7ROU_66I/7ROU_66I_protein.pdb,data/posebusters_benchmark_set/posebusters_benchmark_set/7ROU_66I/7ROU_66I_ligands.sdf
data/posebusters_benchmark_set/posebusters_benchmark_set/8HFN_XGC/8HFN_XGC_protein.pdb,data/posebusters_benchm

# Generate ESM2 embeddings for Proteins
DiffDock -> FASTA -> ESM -> embeddings -> DiffDock

In [ ]:
# This command extracts sequences from protein structures, puts them all in a FASTA file.
!python datasets/esm_embedding_preparation.py \
  --protein_ligand_csv data/moad_for_esm.csv \
  --out_file data/moad_sequences.fasta

# Warnings may appear but that is due to missing residues, common in real-world data.

Traceback (most recent call last):
  File "/content/DiffDock/datasets/esm_embedding_preparation.py", line 10, in <module>
    from .constants import three_to_one
ImportError: attempted relative import with no known parent package


In [ ]:
!cd /content/DiffDock && python -m datasets.esm_embedding_preparation \
  --protein_ligand_csv data/posebusters_for_esm.csv \
  --out_file data/posebusters_sequences.fasta

usage: esm_embedding_preparation.py [-h] [--out_file OUT_FILE]
                                    [--dataset DATASET] [--data_dir DATA_DIR]
esm_embedding_preparation.py: error: unrecognized arguments: --protein_ligand_csv data/posebusters_for_esm.csv


In [ ]:
# Run esm to get embedding
!git clone https://github.com/facebookresearch/esm.git
%cd esm
!pip install -e .
!python scripts/extract.py \
  esm2_t33_650M_UR50D \
  ../data/posebusters_sequences.fasta \
  embeddings_output \
  --repr_layers 33 \
  --include per_tok \
  --truncation_seq_length 4096
!mv embeddings_output ../data/
%cd ..

Cloning into 'esm'...
remote: Enumerating objects: 1511, done.
remote: Counting objects: 100% (807/807), done.
remote: Compressing objects: 100% (215/215), done.
remote: Total 1511 (delta 630), reused 592 (delta 592), pack-reused 704 (from 1)
Receiving objects: 100% (1511/1511), 12.73 MiB | 16.71 MiB/s, done.
Resolving deltas: 100% (953/953), done.
/content/DiffDock/esm
Obtaining file:///content/DiffDock/esm
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for fair-esm (pyproject.toml) ... done
  Created wheel for fair-esm: filename=fair_esm-2.0.1-0.editable-py3-none-any.whl size=18084 sha256=0f04d7198c66956f3f6be65a1f451504d4735d6e4a35439849077efa5f5ec25a
  Stored in directory: /tmp/pip-ephem-wheel-cache-cqscn7qo/wheels/1f/e3/e2/fd59fb727cbdc3ff1bfbbf715bb75d43d9e953ce46351e6aa4
Successfully built fair-esm
  

## Run DiffDock

In [2]:
!echo "# ligand-docking" >> README.md
!git init
!git add README.md
!git commit -m "first commit"
!git branch -M main
!git remote add origin https://github.com/sarahshahrir/ligand-docking.git
!git push -u origin main

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/.git/
Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@f6a0e584c040.(none)')
error: src refspec main does not match any
error: failed to push some refs to 'https://github.com/sarahshahrir/ligand-docking.git